## Imports

In [0]:
from pyspark.sql.functions import date_add, coalesce, lit
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
import boto3

## Read Parameters

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        if aws_secret_key is not None:
            print("AWS secret key retrieved successfully.")
        else:
            print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix

In [0]:
env = get_env_prefix()

In [0]:
Businessdate = spark.sql(f"""                    
                                select PreviousBusinessDay
                                from `{env}-cdh`.reference.ref_calendar
                                where CalendarDate = cast(getdate() as date)
                            """).first()[0]

PreviousBusinessDay = spark.sql(f"""                    
                                select PreviousBusinessDay
                                from `{env}-cdh`.reference.ref_calendar
                                where CalendarDate = '{Businessdate}'
                            """).first()[0]

## Read Tables

In [0]:
DialerActivityDuplicates = spark.sql(f"""
                                        select *
                                        from `{env}-cdh`.reference.dialer_activity_duplicates
                                    """)

DialerActivityDuplicates.createOrReplaceTempView("DialerActivityDuplicates")

In [0]:
previous_extfile_dialerexternal = spark.sql(f"""
                                                select distinct
                                                          CDI
                                                        , CampaignName
                                                        , DateHour
                                                        , CallerPhone
                                                        , AgentName
                                                        , RingTimeDuration
                                                        , TalkTimeDuration
                                                        , HungUpSite
                                                        , CallDisposition
                                                        , AnsweredTime
                                                        , WrapUpTime
                                                        , Name_File_Agent
                                                        , Name_File
                                                        , ID_Records
                                                        , source_BusinessDate
                                                        , LastUpdateDate
                                                        , BusinessDate2
                                                        , ID
                                                from `{env}-cdh`.landed.extfile_dialerexternal
                                                where BusinessDate = '{PreviousBusinessDay}'
                                            """)

previous_extfile_dialerexternal.createOrReplaceTempView("previous_extfile_dialerexternal")

In [0]:
# List all tables in the schema
tables_df = spark.sql(f"SHOW TABLES IN `{env}-reportinputfiles`.internal")

# Filter for tables that match 'DIALER_REPORT_%'
stop_run = tables_df.filter(tables_df.tableName.startswith("dialer_report_")).count() < 1

if stop_run:
    new_extfile_dialerexternal = spark.sql(f"""
                                              SELECT DISTINCT 
                                                        CDI
                                                      , CampaignName
                                                      , DateHour
                                                      , CallerPhone
                                                      , AgentName
                                                      , RingTimeDuration
                                                      , TalkTimeDuration
                                                      , HungUpSite
                                                      , CallDisposition
                                                      , AnsweredTime
                                                      , WrapUpTime
                                                      , Name_File_Agent
                                                      , Name_File
                                                      , ID_Records
                                                      , source_BusinessDate
                                                      , LastUpdateDate
                                                      , BusinessDate2
                                                      , ID
                                                      , '{Businessdate}' BusinessDate
                                                      , getdate() LoadingDateTime
                                              FROM previous_extfile_dialerexternal
                                          """)
    
    new_extfile_dialerexternal.createOrReplaceTempView("new_extfile_dialerexternal")

    table_path = f"`{env}-cdh`.landed.extfile_dialerexternal"
    new_extfile_dialerexternal.write.mode("append").saveAsTable(table_path)

    dbutils.notebook.exit("No table found matching 'dialer_report_%'")

In [0]:
ext_file = tables_df.filter(tables_df.tableName.startswith('dialer_report_')) \
                               .select(tables_df.tableName) \
                               .orderBy(tables_df.tableName)

ext_file_list = [row.tableName for row in ext_file.collect()]

In [0]:
DialerExternal = previous_extfile_dialerexternal
DialerExternal.createOrReplaceTempView("DialerExternal")

for ext_file in ext_file_list:
    dialer_input_file = spark.sql(f"""
                                    select distinct
                                              cast(CDI as string) CDI
                                            , cast(CampaignName as string) CampaignName
                                            , cast(DateHour as timestamp) DateHour
                                            , cast(CallerPhone as string) CallerPhone
                                            , cast(AgentName as string) AgentName
                                            , cast(RingTimeDuration as string) RingTimeDuration
                                            , cast(TalkTimeDuration as string) TalkTimeDuration
                                            , cast(HungUpSite as string) HungUpSite
                                            , cast(CallDisposition as string) CallDisposition
                                            , cast(AnsweredTime as string) AnsweredTime
                                            , cast(WrapUpTime as string) WrapUpTime
                                    from `{env}-reportinputfiles`.internal.`{ext_file}`
                                """)

    dialer_input_file.createOrReplaceTempView("dialer_input_file")

    new_extfile_dialerexternal = spark.sql(f"""
                                              select distinct
                                                        CDI
                                                      , CampaignName
                                                      , DateHour
                                                      , CallerPhone
                                                      , AgentName
                                                      , RingTimeDuration
                                                      , TalkTimeDuration
                                                      , HungUpSite
                                                      , CallDisposition
                                                      , AnsweredTime
                                                      , WrapUpTime
                                                      , cast(NULL as string) Name_File_Agent
                                                      , concat(upper('{ext_file}'), '.txt') Name_File
                                                      , cast(NULL as int) ID_Records
                                                      , to_date(substring('{ext_file}', len('{ext_file}')-7, 8), 'yyyyMMdd') source_BusinessDate
                                                      , getdate() LastUpdateDate
                                                      , cast(NULL as date) BusinessDate2
                                                      , ROW_NUMBER() OVER (ORDER BY 1) AS ID
                                              from dialer_input_file
                                            """)

    new_extfile_dialerexternal.createOrReplaceTempView("new_extfile_dialerexternal")

    DialerExternal = DialerExternal.union(new_extfile_dialerexternal)

DialerExternal = DialerExternal.withColumn("BusinessDate", lit(Businessdate))\
                               .withColumn("LoadingDateTime", lit(datetime.now()))

DialerExternal.createOrReplaceTempView("DialerExternal")

## Deploy

In [0]:
table_path = f"`{env}-cdh`.landed.extfile_dialerexternal"
DialerExternal.write.mode("append").saveAsTable(table_path)

spark.sql(f"DROP TABLE `{env}-reportinputfiles`.internal.`{ext_file}`")

## Modify Table

#### Find duplicate records

In [0]:
duplicates_f = spark.sql(f"""
                            select distinct
                                    COUNT(1) c
                                    , cast(CDI as string) CDI
                                    , CampaignName
                                    , DateHour
                                    , CallerPhone
                            from `{env}-cdh`.landed.extfile_dialerexternal de
                            where not exists (
                                select 1
                                from DialerActivityDuplicates da
                                where da.CallerPhone = de.CallerPhone 
                                and da.CampaignName = de.CampaignName 
                                and coalesce(da.CDI,'') = coalesce(de.CDI,'') 
                                and da.DateHour = de.DateHour
                            )
                            group by all
                            having COUNT(1)>1
                        """)

duplicates_f.createOrReplaceTempView("duplicates_f")

#### Delete records already fixed

In [0]:
affected_rows = spark.sql("""
                            DELETE FROM `{env}-cdh`.landed.extfile_dialerexternal de
                            WHERE EXISTS (
                                SELECT 1
                                FROM DialerActivityDuplicates da
                                WHERE da.CallerPhone = de.CallerPhone
                                AND da.CampaignName = de.CampaignName
                                AND COALESCE(da.CDI, '') = COALESCE(de.CDI, '')
                                AND da.DateHour = de.DateHour
                            )
                        """)

#### Inserting into table dialer_activity_duplicates

In [0]:
affected_rows = spark.sql("""
                            INSERT INTO `{env}-cdh`.reference.dialer_activity_duplicates 
                            (CampaignName, DateHour, CallerPhone, CDI)
                            SELECT distinct 
                                      CampaignName
                                    , DateHour
                                    , CallerPhone
                                    , COALESCE(CDI, '0')
                            FROM duplicates_f
                            WHERE NOT EXISTS (
                                SELECT 1
                                FROM `{env}-cdh`.reference.dialer_activity_duplicates da
                                WHERE da.CallerPhone = duplicates_f.CallerPhone
                                AND da.CampaignName = duplicates_f.CampaignName
                                AND COALESCE(da.CDI, '0') = COALESCE(duplicates_f.CDI, '0')
                                AND da.DateHour = duplicates_f.DateHour
                            )
                        """)

#### Updating DateHour

In [0]:
data_fixed_f = spark.sql("""
                            SELECT DISTINCT
                                      b.id
                                    , date_add(
                                          MILLISECOND
                                          , 10 * rank() OVER (partition by b.CampaignName, b.CDI, b.CallerPhone order by b.id)
                                          , a.DateHour
                                      ) AS DateHour_fixed
                            FROM duplicates_f a
                            INNER JOIN `{env}-cdh`.landed.extfile_dialerexternal b
                            ON a.CallerPhone = b.CallerPhone
                            AND a.CampaignName = b.CampaignName
                            AND COALESCE(a.CDI, '') = COALESCE(b.CDI, '')
                            AND a.DateHour = b.DateHour
                        """)

data_fixed_f.createOrReplaceTempView("data_fixed_f")

In [0]:
affected_rows = spark.sql("""
                            MERGE INTO `{env}-cdh`.landed.extfile_dialerexternal AS target
                            USING data_fixed_f AS source
                            ON target.id = source.id
                            WHEN MATCHED THEN
                            UPDATE SET target.DateHour = source.DateHour_fixed
                        """)